# Standardizing Columns and Types

*Turn inconsistent labels, text, prices, and dates into columns suitable for analysis.*


## Before You Run

In **Add files**, select `retail_sales_raw.csv` from this lesson's supplied assets.
Enter `assets/01_standardizing_columns_and_types` for **Save folder**. The upload creates
missing folders and places the file at
`/home/pyodide/assets/01_standardizing_columns_and_types/retail_sales_raw.csv`.

`/home/pyodide` is the browser's virtual filesystem, not a folder on your
computer. The upload folder is relative to `/home/pyodide`; Python relative
paths use the current directory returned by `os.getcwd()`. Keep the default
working directory in the course site. Locally, run from this chapter folder.
Uploading again to the same complete path replaces that input file.
Saved examples appear under `outputs/01_standardizing_columns_and_types/` in the file browser.


## Inspect the Raw Table

Read the supplied retail orders before deciding what to change. Column
names can contain spaces, text categories can differ in case, and a price
with a dollar sign is text rather than a numeric amount.


In [1]:
import os

import pandas as pd

input_directory = os.path.join("assets", "01_standardizing_columns_and_types")
output_directory = os.path.join("outputs", "01_standardizing_columns_and_types")
input_path = os.path.join(input_directory, "retail_sales_raw.csv")
output_path = os.path.join(output_directory, "standardized_sales.csv")
os.makedirs(output_directory, exist_ok=True)

raw_df = pd.read_csv(input_path, encoding="utf-8")
print("Raw preview:")
print(raw_df.head(3).to_string(index=False))
print("Shape:", raw_df.shape)
print("Column names:", raw_df.columns.tolist())
print("Raw types:")
print(raw_df.dtypes)


Raw preview:
 Order ID Order Date Region   Category Product Name  Quantity Unit Price Customer Type Channel
     1001 2026/01/03  seoul     laptop Notebook Pro       2.0      $1200     corporate  online
     1002 2026/01/05   Busan accessory        Mouse       5.0        $25    individual offline
     1003 2026/01/07   SEOUL   monitor UltraView 27       1.0       $310     corporate  online
Shape: (12, 9)
Column names: ['Order ID', 'Order Date', 'Region ', 'Category', 'Product Name', 'Quantity', 'Unit Price', 'Customer Type', 'Channel']
Raw types:
Order ID           int64
Order Date           str
Region               str
Category             str
Product Name         str
Quantity         float64
Unit Price           str
Customer Type        str
Channel              str
dtype: object


There are 12 rows and 9 columns. `Region ` contains a trailing space, `Unit Price` is text, and `Order Date` has not yet been parsed as dates.


## Standardize Names and Category Text

`.str` applies string operations across labels or a text column.
`.strip()` removes surrounding whitespace, `.lower()` makes names lowercase,
and `.replace(" ", "_", regex=False)` replaces literal spaces.
For these prepared categories, `.title()` gives consistent capitalization.


In [2]:
sales_df = raw_df.copy()
sales_df.columns = (
    sales_df.columns.str.strip().str.lower().str.replace(" ", "_", regex=False)
)
for column in ["region", "category", "customer_type", "channel"]:
    sales_df[column] = sales_df[column].str.strip().str.title()
sales_df["product_name"] = sales_df["product_name"].str.strip()

print("Standardized column names:", sales_df.columns.tolist())
print("Region labels:", sorted(sales_df["region"].unique()))
print("Customer types:", sorted(sales_df["customer_type"].unique()))


Standardized column names: ['order_id', 'order_date', 'region', 'category', 'product_name', 'quantity', 'unit_price', 'customer_type', 'channel']
Region labels: ['Busan', 'Daegu', 'Incheon', 'Seoul']
Customer types: ['Corporate', 'Individual']


Whitespace and case variants of Seoul now share the label `Seoul`. The regions are Busan, Daegu, Incheon, and Seoul; customer types are Corporate and Individual.


## Convert Prices, Quantities, and Dates

`pd.to_numeric()` converts numeric text. First remove the literal `$`
from prices. `errors="coerce"` turns unparseable values into missing values,
which must be inspected rather than assumed correct.

`pd.to_datetime(..., format="mixed")` parses each date's format. These
inputs use unambiguous year-first dates with either slashes or hyphens.


In [3]:
sales_df["unit_price"] = pd.to_numeric(
    sales_df["unit_price"].str.replace("$", "", regex=False), errors="coerce"
)
sales_df["quantity"] = pd.to_numeric(sales_df["quantity"], errors="coerce")
sales_df["order_date"] = pd.to_datetime(
    sales_df["order_date"], format="mixed", errors="coerce"
)

print("Converted preview:")
print(sales_df[["order_id", "order_date", "region", "quantity", "unit_price"]].head(5))
print("Converted types:")
print(sales_df[["order_date", "quantity", "unit_price"]].dtypes)
print("Missing values after conversion:")
print(sales_df[["order_date", "quantity", "unit_price"]].isna().sum())
print("Rows after conversion:", len(sales_df))


Converted preview:
   order_id order_date   region  quantity  unit_price
0      1001 2026-01-03    Seoul       2.0        1200
1      1002 2026-01-05    Busan       5.0          25
2      1003 2026-01-07    Seoul       1.0         310
3      1004 2026-01-12  Incheon       1.0         980
4      1005 2026-01-18    Daegu       NaN          45
Converted types:
order_date    datetime64[us]
quantity             float64
unit_price             int64
dtype: object
Missing values after conversion:
order_date    0
quantity      1
unit_price    0
dtype: int64
Rows after conversion: 12


All 12 rows remain, and quantity still has one missing value. Dates and prices have zero missing values after parsing. Type conversion and record cleaning are separate decisions.


## Save and Verify the Standardized Table

CSV stores text rather than pandas dtype metadata. On reload, request
date parsing with `parse_dates=["order_date"]`. `date_format` writes dates
consistently as year-month-day. No rows are removed in this lesson.


In [4]:
sales_df.to_csv(output_path, index=False, encoding="utf-8", date_format="%Y-%m-%d")
reopened_df = pd.read_csv(output_path, encoding="utf-8", parse_dates=["order_date"])

print("Saved file:", output_path)
print("Reopened preview:")
print(reopened_df[["order_id", "order_date", "region", "quantity", "unit_price"]].head(3))
print("Round trip matches:", reopened_df.equals(sales_df))


Saved file: outputs/01_standardizing_columns_and_types/standardized_sales.csv
Reopened preview:
   order_id order_date region  quantity  unit_price
0      1001 2026-01-03  Seoul       2.0        1200
1      1002 2026-01-05  Busan       5.0          25
2      1003 2026-01-07  Seoul       1.0         310
Round trip matches: True


The saved table matches after explicitly restoring the date column. It still has 12 rows; missing quantities and repeated orders are handled in later lessons.


Standardize labels and categories, then convert numeric and date columns. Inspect missing values introduced or retained by conversion before summarizing.

[Date conversion](https://pandas.pydata.org/docs/reference/api/pandas.to_datetime.html)
